# 03 - Digital Twin: LSTM Model Training

This notebook walks through the complete workflow for training an LSTM
neural network to forecast solar power generation using the DERIM
digital twin module.

## Objectives

1. Load and preprocess the sample solar data.
2. Create sliding-window sequences for the LSTM.
3. Train the model and monitor loss convergence.
4. Evaluate the model against baseline forecasters.
5. Visualise predictions vs actual values.
6. Save the trained model for deployment.

---

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load and Preprocess Data

In [ ]:
from derim.digital_twin.trainer import Trainer

trainer = Trainer()

# Load the sample CSV
df = trainer.load_csv('../data/sample_solar_data.csv')
print(f'Loaded {len(df)} rows')
df.head()

In [ ]:
# Preprocess: fill missing values, clip negatives
df = trainer.preprocess(df, target_column='power_kw')
print(f'After preprocessing: {len(df)} rows')

# Plot the target variable
plt.figure(figsize=(14, 4))
plt.plot(df['timestamp'], df['power_kw'], linewidth=0.5)
plt.title('Solar Power Output (preprocessed)')
plt.ylabel('Power (kW)')
plt.xlabel('Time')
plt.tight_layout()
plt.show()

## 2. Train/Test Split

We use the first 25 days for training and the last 5 days for testing.

In [ ]:
split_date = df['timestamp'].min() + pd.Timedelta(days=25)
train_df = df[df['timestamp'] < split_date].copy()
test_df = df[df['timestamp'] >= split_date].copy()

print(f'Training set: {len(train_df)} rows ({train_df.timestamp.min()} to {train_df.timestamp.max()})')
print(f'Test set:     {len(test_df)} rows ({test_df.timestamp.min()} to {test_df.timestamp.max()})')

## 3. Train the LSTM Model

We use the `Trainer` class which wraps the `LSTMForecaster` for a
streamlined training experience.

In [ ]:
from derim.digital_twin.models.lstm_forecaster import LSTMForecaster
from derim.config import Settings

# Configure for notebook training
import os
os.makedirs('../saved_models', exist_ok=True)

settings = Settings(
    model_save_dir='../saved_models',
    lstm_epochs=30,       # Reduce for demo; use 50-100 in production
    lstm_batch_size=32,
    lstm_sequence_length=96,  # 24 hours of 15-min data
    forecast_horizon_hours=24,
)

forecaster = LSTMForecaster(settings=settings)

print('Starting LSTM training...')
metrics = forecaster.train(
    device_id='solar-inv-001',
    data=train_df,
    target_column='power_kw',
    epochs=30,
    batch_size=32,
    learning_rate=0.001,
    validation_split=0.2,
)

print(f"\nTraining complete!")
print(f"Best validation loss: {metrics['best_val_loss']:.6f}")

## 4. Training Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(metrics['history']['train_loss'], label='Training Loss', linewidth=2)
ax.plot(metrics['history']['val_loss'], label='Validation Loss', linewidth=2)
ax.set_title('LSTM Training Loss Curves', fontsize=13)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 5. Generate Predictions and Compare

In [ ]:
# Use last 96 values from training set as input
input_values = train_df['power_kw'].values[-96:].tolist()

predictions = forecaster.predict(
    device_id='solar-inv-001',
    horizon_hours=24,
    recent_values=input_values,
)

pred_power = [p.power_kw for p in predictions]
pred_times = [p.timestamp for p in predictions]

# Get actual values for comparison
actual_24h = test_df['power_kw'].values[:96]
actual_times = test_df['timestamp'].values[:96]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(actual_24h)), actual_24h, label='Actual', linewidth=2, color='#2ECC71')
ax.plot(range(len(pred_power)), pred_power, label='LSTM Prediction', linewidth=2, color='#E74C3C', linestyle='--')
ax.set_title('24-Hour Solar Power Forecast vs Actual', fontsize=13)
ax.set_xlabel('Time Step (15-min intervals)')
ax.set_ylabel('Power (kW)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Evaluate Against Baselines

In [ ]:
from derim.digital_twin.models.baseline import PersistenceForecaster, MovingAverageForecaster

# Persistence baseline
persistence = PersistenceForecaster()
pers_preds = persistence.predict(input_values, horizon_hours=24)
pers_power = [p.power_kw for p in pers_preds]

# Moving average baseline
ma = MovingAverageForecaster(window_size=96)
ma_preds = ma.predict(input_values, horizon_hours=24)
ma_power = [p.power_kw for p in ma_preds]

# Compute metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

n = min(len(actual_24h), len(pred_power), len(pers_power), len(ma_power))

results = {
    'Model': ['LSTM', 'Persistence', 'Moving Average'],
    'MAE (kW)': [
        round(mean_absolute_error(actual_24h[:n], pred_power[:n]), 4),
        round(mean_absolute_error(actual_24h[:n], pers_power[:n]), 4),
        round(mean_absolute_error(actual_24h[:n], ma_power[:n]), 4),
    ],
    'RMSE (kW)': [
        round(np.sqrt(mean_squared_error(actual_24h[:n], pred_power[:n])), 4),
        round(np.sqrt(mean_squared_error(actual_24h[:n], pers_power[:n])), 4),
        round(np.sqrt(mean_squared_error(actual_24h[:n], ma_power[:n])), 4),
    ],
}

results_df = pd.DataFrame(results)
print('=== Model Comparison ===')
results_df

## 7. Digital Twin Simulator

Use the `Simulator` class to compare predictions with actuals and detect anomalies.

In [ ]:
from derim.digital_twin.simulator import Simulator

sim = Simulator(device_id='solar-inv-001', settings=settings)
result = sim.compare(
    actual_values=actual_24h[:n].tolist(),
    predicted_values=pred_power[:n],
)

print(f'MAE:  {result.mae} kW')
print(f'RMSE: {result.rmse} kW')
print(f'Anomaly threshold: {result.anomaly_threshold} kW')
print(f'Anomalous points: {len(result.anomaly_indices)}')

# Plot residuals
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(result.residuals)), result.residuals, color='#3498DB', alpha=0.7, width=1.0)
ax.axhline(result.anomaly_threshold, color='red', linestyle='--', label=f'Anomaly threshold (±{result.anomaly_threshold:.2f} kW)')
ax.axhline(-result.anomaly_threshold, color='red', linestyle='--')
ax.set_title('Prediction Residuals (Actual - Predicted)', fontsize=13)
ax.set_xlabel('Time Step')
ax.set_ylabel('Residual (kW)')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the full digital twin training pipeline:

1. Data loading and preprocessing via the `Trainer` class.
2. LSTM model training with loss monitoring.
3. 24-hour ahead forecasting and comparison with baselines.
4. Digital twin simulation with anomaly detection.

The trained model is saved to `saved_models/` and can be used by the
REST API's `/forecast/{device_id}` endpoint.